In [1]:
from pathlib import Path
import sys
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

project_root = Path().resolve().parent
sys.path.append(str(project_root))

from scripts.rq3_function_lib import show_median_price_heatmap_per_region, mann_whitney_test_border_prices, show_border_price_difference, perform_matched_panel_regression_autobahn_stations
from scripts.rq3_function_lib import plot_yearly_autobahn_premium_line, plot_station_price_map, plot_autobahn_premium_barchart
from scripts.rq3_function_lib import perform_wilcoxon_variance_test_on_autobahn, plot_wilcoxon_results_loolipop

In [2]:
region_price_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/regions_avg_prices_per_year')
region_path = Path(r'/Users/sebastian/data-science-projekt/plz_leitregionen.csv')
border_stations_file = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/lower_border_stations.csv')
median_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/station_daily_mean_and_median_by_month')
non_autobahn_border_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/lower_non_autobahn_border_stations.csv')
stations_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/stations.csv')
panel_result_path = Path(r'/Users/sebastian/data-science-projekt/rq_results/rq3_panel_summary.csv')
residual_error_path = Path(r'/Users/sebastian/data-science-projekt/rq_results/rq3_residual_panel_error.parquet')
wilcoxon_result_path = Path(r'/Users/sebastian/data-science-projekt/rq_results/rq3_wilcoxon.csv')


# Regional price differences and price stability

### Do the gas prices in border regions differ from comparable non-border regions?

### How much more do you have to pay for fuel on Autobahn gas stations really?

### Are the gas prices at Autobahn stations more volatile than the prices at normal gas stations?

In [3]:
year = 2025
fuel_type = "diesel"

In [4]:
show_median_price_heatmap_per_region(region_price_path, region_path, year, fuel_type)

As we can see in this map, the fuel prices in Germany differ from region to region and over time. This map plots the development of monthly median prices of a selected fuel type and year for each post code "Leitregion" (for a map of these regeions, see RQ9 notebook). In the following, we want to have a deeper look into the German border regions and see if the prices in these regions differ from prices in comparable non-border regions. After that, we will have a look at the autobahn gas stations to see how much more expensive they are and if their prices have a different volatility compared to normal gas stations. 

### How does the price at the stations close to the german border (<=8km dist) differ from other stations in their surrounding area (8-25km from border)?

In [5]:

border_stations = pl.read_csv(border_stations_file, schema_overrides = {"post_code": pl.Utf8})

In [6]:
fig = px.scatter_map(border_stations,
                    lat = "latitude",
                    lon= "longitude",
                    color = "border_region",
                    hover_name = "neighbour_country",
                    hover_data = "neighbour_country",
                    center = {"lat": 51.16, "lon": 10.45},
                    zoom = 4,
                    map_style = "open-street-map",
                    title = "Border and surrounding stations in germany")

fig.update_layout(margin = {"r":0,"t":50,"l":0,"b":0})
fig.update_traces(marker = dict(size = 15, opacity = 1))
fig.show()

This map shows all border region stations and the stations from the surrounding area. In the following test, we grouped the gas stations by their nearest bordering country to get separate  results for each country neighbouring Germany.

### Is fuel in border regions more/less expensive compared to non-border region stations?
This test examines whether fuel prices at stations **close to the border** differ
from prices at stations in the **surrounding region**.

The analysis is based on station-level median prices. For each station, the fuel-specific
median price is first calculated either **within each year** or **across all years**.
This ensures that each station contributes one representative value to the comparison.

For a given neighbouring country, two independent groups are then compared:

- **Border group**: stations that are **up to 8km away** from the next border
- **Surrounding group**: stations that are **8-25km away** from the next border

The statistical test used is the **two-sided Mann–Whitney U test**. In general,
this test evaluates whether two independent samples come from the same distribution
without requiring a normality assumption.

Let $X_1, \dots, X_n$ denote the station-level median prices in the border region
and $Y_1, \dots, Y_m$ the station-level median prices in the surrounding region.
The null and alternative hypotheses are

$$
H_0: F_X = F_Y
$$

and

$$
H_1: F_X \neq F_Y
$$

where $F_X$ and $F_Y$ are the distribution functions of the two groups.

In addition to the test statistic and p-value, the method reports the group medians:

$$
\tilde{P}_{\text{border}}
\qquad \text{and} \qquad
\tilde{P}_{\text{surrounding}}
$$

and their difference

$$
\text{Price difference}
=
\tilde{P}_{\text{border}} - \tilde{P}_{\text{surrounding}}
$$

A negative value indicates that border stations are cheaper on average in median terms,
while a positive value indicates that border stations are more expensive.

The test is carried out both:

- **overall**, using station-level medians across all years, and
- **yearly**, using station-level medians within each year.

A result is marked as statistically significant if the p-value is below 0.05.
Only country-group comparisons with at least **5 stations in each region** are tested.

In [7]:
overall_result_df, yearly_result_df = mann_whitney_test_border_prices(median_path, border_stations_file, "diesel")
print("=== absolute results (over all years) ===")
display(overall_result_df)

print("\n=== yearly result (excerpt) ===")
display(yearly_result_df.head(15))

=== absolute results (over all years) ===


,Country,N_border,N_surrounding,Median_border,Median_surrounding,Price_difference,p_value,Significant (5%)
0,France,171,366,1.329,1.317,0.012,8.538706e-07,True
1,Czechia,55,237,1.289,1.299,-0.010,3.238376e-01,False
2,Poland,40,60,1.300,1.299,0.001,8.735863e-01,False
3,Belgium,34,62,1.309,1.299,0.010,2.331846e-01,False
4,Denmark,33,21,1.279,1.279,0.000,6.085410e-01,False
5,Netherlands,257,397,1.309,1.299,0.010,2.954649e-06,True
6,Switzerland,54,54,1.349,1.326,0.023,4.726668e-03,True
7,Austria,91,242,1.339,1.329,0.010,6.066301e-05,True



=== yearly result (excerpt) ===


,year,Country,Median_border,Median_surrounding,Price_difference,p_value,Significant (5%),N_border,N_surrounding
0,2014,France,1.354,1.349,0.005,9.948110e-04,True,157,339
1,2015,France,1.182,1.174,0.008,2.482858e-04,True,163,346
2,2016,France,1.099,1.091,0.008,9.512268e-05,True,164,349
3,2017,France,1.179,1.164,0.015,8.069883e-07,True,168,360
4,2018,France,1.329,1.304,0.025,4.315625e-12,True,171,355
5,2019,France,1.289,1.274,0.015,2.614502e-07,True,169,350
6,2020,France,1.094,1.084,0.010,1.002480e-03,True,169,346
7,2021,France,1.369,1.364,0.005,2.518849e-01,False,168,339
8,2022,France,1.977,1.969,0.008,2.897243e-01,False,165,332
9,2023,France,1.709,1.709,0.000,9.522379e-01,False,162,326


#### Results

The overall results suggest that fuel prices at border stations are **not uniformly lower**
than in the surrounding region. Instead, the pattern differs across neighbouring countries.

Across **all years combined**, statistically significant differences are found for
**France, Austria, the Netherlands, and Switzerland**. In all four cases, the estimated
price difference is **positive**, meaning that median prices at border stations are
slightly **higher** than in the surrounding region. The largest difference appears for
**Switzerland** with about **0.023 EUR/L**, followed by **France** (**0.012 EUR/L**),
and **Austria** and the **Netherlands** (both around **0.010 EUR/L**).

A possible explanation for this could be, that **Austria** and **France** are classical european transit countries where major motorways cross through. For **Switzerland** and the **Netherlands** an explanation could be an inverted "fuel tourism" from these countries, because their fuel price level is much higher than in Germany. 

For **Denmark, Poland, Czechia, and Belgium**, the p-values are above 0.05, so there is
no statistically significant evidence of a systematic difference between border and
surrounding stations in the pooled comparison. In the case of **Czechia**, the estimated
difference is slightly **negative** (-0.010 EUR/L), which would indicate cheaper border
stations, but this difference is not statistically significant.

Because of the too small sample size, **Luxembourg** isn't considered in the test.

Overall, the results do **not** support a general conclusion that border stations are
cheaper. Where significant differences are found, they more often indicate **higher**
prices in the border region, although the estimated magnitudes are economically fairly small.

Note that we only calculated a test for diesel as an example. When performing the same test for e5 and e10, the results where pretty similar.

When first performing the test, we had defined broader border distances (0-15km and 15-50km). This test concluded, that all border regions where statistically significant more expensive. When revising the methodology, we found out that with these larger distances, many parts of the surrounding (control) regions where urbanized compared to the rural border regions. To mitigate this distortion, we decided to change the region definitions to the one used above.

To rule out other possible test design errors, we now perform the same test, but this time with the autobahn stations filtered out. After that, we check if the brand structure differs in the border vs. surrounding region to also rule out this factor.

In [8]:

overall_result_df, yearly_result_df = mann_whitney_test_border_prices(median_path, non_autobahn_border_path, "diesel")

print("=== absolute results (over all years) ===")
display(overall_result_df)

print("\n=== yearly result (excerpt) ===")
display(yearly_result_df.head(15))

=== absolute results (over all years) ===


,Country,N_border,N_surrounding,Median_border,Median_surrounding,Price_difference,p_value,Significant (5%)
0,Austria,86,236,1.339,1.329,0.010,0.000188,True
1,Switzerland,54,53,1.349,1.329,0.020,0.005640,True
2,Belgium,34,62,1.309,1.299,0.010,0.233185,False
3,France,160,364,1.329,1.314,0.015,0.000006,True
4,Czechia,55,230,1.289,1.299,-0.010,0.479434,False
5,Netherlands,255,393,1.309,1.299,0.010,0.000001,True
6,Poland,39,58,1.301,1.299,0.002,0.853299,False
7,Denmark,33,21,1.279,1.279,0.000,0.608541,False



=== yearly result (excerpt) ===


,year,Country,Median_border,Median_surrounding,Price_difference,p_value,Significant (5%),N_border,N_surrounding
0,2014,Austria,1.359,1.359,0.000,4.552153e-02,True,73,209
1,2015,Austria,1.189,1.179,0.010,2.928403e-03,True,78,223
2,2016,Austria,1.099,1.089,0.010,9.159282e-04,True,81,231
3,2017,Austria,1.184,1.169,0.015,2.833606e-05,True,84,233
4,2018,Austria,1.314,1.299,0.015,5.223490e-05,True,84,235
5,2019,Austria,1.299,1.287,0.012,1.523891e-04,True,83,232
6,2020,Austria,1.109,1.089,0.020,1.912970e-09,True,82,228
7,2021,Austria,1.389,1.379,0.010,9.521161e-04,True,82,226
8,2022,Austria,2.019,2.014,0.005,3.199128e-01,False,82,220
9,2023,Austria,1.759,1.759,0.000,1.501365e-01,False,81,214


As we can see, the **overall test result** is still the same. 

**Switzerland, the Netherlands, France and Austria** have statistically significant results, while **Belgium, Denmark, Czechia and Poland** are still statistically insignificant.

Although the price differences have changed a bit, the result still holds.

To visualize the (missing) price difference, we plot the median prices for a year and for a specific border region, in this case **Czechia**. 

For most of the years, the distributions are pretty similar as we can see. This visually confirmes the results for the above test for Czechia.

In [21]:
fuel = "diesel"
country = "Czechia"
 

show_border_price_difference(median_path, non_autobahn_border_path, fuel, country)

Now we check if the border region non autobahn stations are brand stations or non brand stations.

We manually decided the brands that are categorized as "premium brands".

In [10]:

stations_df = pl.read_csv(stations_path, schema_overrides={"post_code": pl.Utf8})
no_autobahn_border_df = pl.read_csv(non_autobahn_border_path, schema_overrides = {"post_code": pl.Utf8})

brand_border_region_df = (stations_df.join(no_autobahn_border_df,
                                           how = "inner",
                                           on = "uuid"))

In [11]:

premium_brands = ["ARAL", "SHELL", "JET", "TOTAL", "TOTAL ENERGIES", "ESSO", "AVIA",
    "HEM", "HOYER", "ORLEN", "Q1", "STAR", "RAIFFEISEN", "AGIP",
    "ENI", "OMV", "OIL!", "WESTFALEN"]

#categorize each station into brand or non brand
brand_df = (brand_border_region_df.with_columns(
    pl.col("brand").str.to_uppercase().str.strip_chars().alias("clean_brands")
).with_columns(
    pl.when(pl.col("clean_brands").is_in(premium_brands)).then(pl.lit("brand"))
    .otherwise(pl.lit("non brand")).alias("brand_category")
))
#aggregate and calculate percents
percent_df = (brand_df.group_by(["border_region", "brand_category"])
              .agg(pl.len().alias("counter"))
              .with_columns((pl.col("counter") / pl.col("counter").sum().over("border_region") * 100)
                            .round(2).alias("percentage")))
pivot_df = (percent_df.pivot(on = "brand_category",
                             index = "border_region",
                             values = "percentage"))

print("=== brand structure: border vs. surrounding regions ===")
print(pivot_df)

=== brand structure: border vs. surrounding regions ===
shape: (2, 3)
┌──────────────────────┬───────┬───────────┐
│ border_region        ┆ brand ┆ non brand │
│ ---                  ┆ ---   ┆ ---       │
│ str                  ┆ f64   ┆ f64       │
╞══════════════════════╪═══════╪═══════════╡
│ Border (0-8km)       ┆ 61.58 ┆ 38.42     │
│ Surrounding (8-25km) ┆ 61.31 ┆ 38.69     │
└──────────────────────┴───────┴───────────┘


As we can see, the brand structure doesn't really differ. We can rule out this as an influence factor for our test results. 

### How much more expensive are Autobahn gas stations really?

The next part of the questions looks for the price differences between autobahn stations and regular stations. 

For that we use a matched panel regression that examines whether Autobahn stations have systematically higher fuel prices than nearby non-Autobahn stations. 

The model absorbs fixed regional and time effects. Additionaly, to mitigate brand effects, we categorize each station into brand, non-brand/free and unknown.

The models dependent variable is the daily station-level fuel price for a selected
fuel type and summary statistic. Letting $y_{i,t}$ denote the observed fuel
price of station $i$ on date $t$, the outcome in this analysis is one of

$$
y_{i,t} \in
\left\{
\text{diesel}_{i,t},
\text{e5}_{i,t},
\text{e10}_{i,t}
\right\}
$$

measured either as the daily **mean** or daily **median** price.

To make Autobahn and non-Autobahn stations geographically comparable, each
Autobahn station is matched with up to **5 nearest non-Autobahn stations**
within a maximum distance of **50 km**. This creates local comparison groups that reduce the bias from regional price differences.

The regression model from AbsorbingLS can be written in the general form

$$
y_i = x_i \beta + z_i \gamma + \epsilon_i
$$

where $y_i$ is the fuel price outcome, $x_i$ the regressor of
interest, and $z_i$ collects the fixed effects.

In our application, this corresponds to

$$
y_{i,t}
=
\beta \cdot \text{autobahn}_i
+ \theta^\top \text{brand\_category}_i
+ \alpha_m
+ \delta_t
+ \varepsilon_{i,t}
$$

where $\text{autobahn}_i$ is an indicator equal to 1 if station $i$ is
located on the Autobahn, $\text{brand\_category}_i$ captures the station's
brand classification, $\alpha_m$ denotes match-set (regional) fixed effects, and
$\delta_t$ denotes date fixed effects.

$\beta$ is the coefficient of interest. It captures the average price
difference between Autobahn and non-Autobahn stations after controlling for
local geographic proximity through the matched sets, for common daily shocks
through date fixed effects, and for systematic brand differences through brand
controls.

The model is estimated separately for each year and for each combination of
fuel type (**diesel**, **e5**, **e10**) and price statistic (**mean**,
**median**). Standard errors are clustered at the **station level** to account
for serial dependence in repeated observations of the same station over time.

We iterate over each fuel type, first performing the test on the mean prices and then to check the results, again over the median prices.

In [12]:
""" autobahn_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/autobahn_stations.csv')

fuel_types = ["diesel", "e5", "e10"]
statistics = ["mean", "median"]

test_summaries = []
analysis_panel_list = []
residuals_list = []

for fuel in fuel_types:
    for stat in statistics:
        
        res, panel, residuals = perform_matched_panel_regression_autobahn_stations(median_path,stations_path,autobahn_path,fuel,stat,return_residuals=True)
        test_summaries.append(res)
        analysis_panel_list.append(panel)
        residuals_list.append(residuals)



residual_df = pd.concat(residuals_list, ignore_index=True)
summary_df = pd.concat(test_summaries, ignore_index = True)
print("\nSummary:")
summary_df
residual_df.to_parquet(r'/Users/sebastian/data-science-projekt/rq_results/rq3_residual_panel_error.parquet')
summary_df.to_csv(r'/Users/sebastian/data-science-projekt/rq_results/rq3_panel_summary.csv')
 """

' autobahn_path = Path(r\'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/autobahn_stations.csv\')\n\nfuel_types = ["diesel", "e5", "e10"]\nstatistics = ["mean", "median"]\n\ntest_summaries = []\nanalysis_panel_list = []\nresiduals_list = []\n\nfor fuel in fuel_types:\n    for stat in statistics:\n\n        res, panel, residuals = perform_matched_panel_regression_autobahn_stations(median_path,stations_path,autobahn_path,fuel,stat,return_residuals=True)\n        test_summaries.append(res)\n        analysis_panel_list.append(panel)\n        residuals_list.append(residuals)\n\n\n\nresidual_df = pd.concat(residuals_list, ignore_index=True)\nsummary_df = pd.concat(test_summaries, ignore_index = True)\nprint("\nSummary:")\nsummary_df\nresidual_df.to_parquet(r\'/Users/sebastian/data-science-projekt/rq_results/rq3_residual_panel_error.parquet\')\nsummary_df.to_csv(r\'/Users/sebastian/data-science-projekt/rq_results/rq3_panel_summary.csv\')\n '

In [13]:
summary_df = pd.read_csv(panel_result_path)
summary_df.head(15)

,Unnamed: 0,year,fuel_type,statistic,autobahn_coef,standard_error,p_value,ci_low,ci_high,n_observed
0,0,2014,diesel,mean,0.043297,0.001647,0.0,0.040070,0.046525,377130
1,1,2015,diesel,mean,0.051106,0.001983,0.0,0.047219,0.054994,685629
2,2,2016,diesel,mean,0.068193,0.002751,0.0,0.062802,0.073584,707278
3,3,2017,diesel,mean,0.080873,0.003013,0.0,0.074967,0.086778,716043
4,4,2018,diesel,mean,0.121318,0.003777,0.0,0.113914,0.128721,720505
5,5,2019,diesel,mean,0.157382,0.005040,0.0,0.147503,0.167261,711279
6,6,2020,diesel,mean,0.181175,0.006328,0.0,0.168773,0.193578,679417
7,7,2021,diesel,mean,0.193057,0.006648,0.0,0.180027,0.206087,667035
8,8,2022,diesel,mean,0.184888,0.006655,0.0,0.171844,0.197931,661864
9,9,2023,diesel,mean,0.240935,0.008872,0.0,0.223546,0.258324,652809


plotting the results

In [14]:
plot_yearly_autobahn_premium_line(summary_df)


In [15]:
plot_autobahn_premium_barchart(summary_df)

#### Results:

The results show a **positive and statistically significant Autobahn premium for fuel prices in every year** from 2014 to 2026. The estimated coefficient rises from about **€0.04 per liter in 2014** to about **€0.29 per liter in 2026** (), which suggests that fuel prices at Autobahn stations were consistently higher than at matched non-Autobahn stations and that this price gap increased substantially over time.

The increase is especially visible from **2018 onward**, with another strong rise after **2022**. Since the confidence intervals remain clearly above zero in all years, the estimated premium appears to be very robust across specifications and years.

The regression suggests that even after controlling for **local matching structure**, **date fixed effects**, and **brand-category differences**, Autobahn stations tend to charge a noticeable price premium relative to comparable nearby stations.

Possible explanations for this may include:
- **weaker competitive pressure on the Autobahn**, since around 93% of the Autobahn service stations are owned by the (formerly government-owned and now private) Tank & Rast GmbH
- auction-based **supply and distribution rights** that oil companies have to obtain could contribute to higher fuel prices 
- the sharp increase in the later years may also be related to the broader energy market disruptions following the **war in Ukraine** and generally higher operating and service costs

At the same time, these regressions identify a **systematic association**, not a definitive causal mechanism. The proposed explanations are therefore plausible interpretations of the observed premium, but they are **not directly tested by this model**.





### Are the gas prices at Autobahn stations more volatile than the prices at normal gas stations?
Now we perform a wilcoxon signed rank test to see wether the autobahn station prices differ from normal station prices in volatility.

For this test, we examine whether the **residual price volatility** differs between
**Autobahn stations** and their matched **non-Autobahn controls** after the
fixed effects on the price have been removed in the panel
regression above.

The input for the test is the set of regression residuals
$\hat{\varepsilon}_{i,t}$ saved from the panel regession model. These residuals
represent the part of station-level fuel prices that remains unexplained after
controlling for the Autobahn factor, brand effects, regional fixed
effects, and date fixed effects.

For each station $i$ in year $y$, volatility is summarized from the residual
series using one of two volatility measures. If the robust option is selected,
volatility is measured by the **median absolute deviation**

$$
\text{median-absolute-deviation}_{i,y}
=
\operatorname{median}_{t \in y}
\left|
\hat{\varepsilon}_{i,t}
-
\operatorname{median}_{s \in y}(\hat{\varepsilon}_{i,s})
\right|
$$

Alternatively, volatility can be measured by the **standard deviation**

$$
\text{SD}_{i,y}
=
\sqrt{
\frac{1}{n_{i,y}-1}
\sum_{t \in y}
\left(
\hat{\varepsilon}_{i,t}
-
\bar{\hat{\varepsilon}}_{i,y}
\right)^2
}
$$

where $n_{i,y}$ is the number of residual observations for station $i$ in year
$y$. We will only consider stations with at least **30 observations** for the
volatility comparison.

To obtain a matched comparison, the volatility of the Autobahn station in match
set $m$ is compared with the average volatility of its matched non-Autobahn
control stations. Letting $V^T_{m,y}$ denote the Autobahn-station volatility and
$\bar{V}^C_{m,y}$ the mean control volatility in the same match set and year, we
define the paired difference as

$$
d_{m,y} = V^T_{m,y} - \bar{V}^C_{m,y}
$$

We implemented a two sided test for paired samples. In the present application, this is appropriate
because volatility is compared **within matched sets**, so the comparison is
based on paired Autobahn-control differences rather than on two independent
samples.

Following the formal definition, the two-sided test evaluates
whether the distribution of the paired differences $d_{m,y}$ is symmetric about
zero. The null and alternative hypotheses can be written as

$$
H_0: d_{m,y} \text{ is symmetrically distributed around } 0
$$

$$
H_1: d_{m,y} \text{ is not symmetrically distributed around } 0
$$

For the test, the nonzero absolute differences $|d_{m,y}|$ are ranked, their
original signs are reattached, and the Wilcoxon statistic is constructed from
the signed ranks. In the two-sided case, the test statistic is

$$
T = \min(W^+, W^-)
$$

where $W^+$ is the sum of ranks associated with positive differences and $W^-$
is the sum of ranks associated with negative differences. In our function,
pairs with $d_{m,y} = 0$ are removed before the test, so only non-zero matched
differences contribute to the statistic.

In our application, a **positive median difference** indicates that Autobahn
stations exhibit higher residual volatility than their matched non-Autobahn
controls in the same year. A **small p-value** suggests that the paired
volatility differences are not centered symmetrically around zero, which is
evidence of a systematic difference in residual volatility between the two
groups.


In [16]:
residual_df = pd.read_parquet(residual_error_path)

In [17]:
""" wilcoxon_df = perform_wilcoxon_variance_test_on_autobahn(residual_df,measure="mad")

print(wilcoxon_df.head(15)) 
wilcoxon_df.to_csv(r'/Users/sebastian/data-science-projekt/rq_results/rq3_wilcoxon.csv') """

' wilcoxon_df = perform_wilcoxon_variance_test_on_autobahn(residual_df,measure="mad")\n\nprint(wilcoxon_df.head(15)) \nwilcoxon_df.to_csv(r\'/Users/sebastian/data-science-projekt/rq_results/rq3_wilcoxon.csv\') '

In [18]:
wilcoxon_df = pd.read_csv(wilcoxon_result_path)

wilcoxon_df

,Unnamed: 0,year,measure,n_pairs,mean_test_volatility,mean_control_volatility,median_difference,wilcoxon_stat,p_value
0,0,2014,mad,294,0.011654,0.011069,0.000352,18788.0,4.725826e-02
1,1,2015,mad,299,0.015624,0.011839,0.002557,7142.0,1.712751e-24
2,2,2016,mad,301,0.014081,0.011854,0.001489,12326.0,5.929404e-12
3,3,2017,mad,307,0.014609,0.012092,0.000848,15461.0,1.490211e-07
4,4,2018,mad,374,0.019972,0.016912,0.001742,23332.0,2.058752e-08
5,5,2019,mad,309,0.018834,0.013823,0.002479,11141.0,3.711624e-16
6,6,2020,mad,305,0.019792,0.014994,0.003261,7691.0,3.404717e-24
7,7,2021,mad,298,0.017602,0.013051,0.002046,8897.0,2.556332e-19
8,8,2022,mad,296,0.036291,0.021432,0.014303,1791.0,1.057824e-42
9,9,2023,mad,300,0.030703,0.017223,0.009268,2305.0,2.061131e-41


In [19]:
plot_wilcoxon_results_loolipop(wilcoxon_df)

#### Results:

The test results show that **Autobahn stations have statistically significant higher residual price volatility than matched non-Autobahn stations in every year** from 2014 to 2026. This is visible in the fact that the **median difference is positive in all years**, which means that the volatility measure based on the residuals is consistently larger for Autobahn stations than for their matched non-Autobahn control groups.

The effect is already present in the early years, but it becomes much stronger from **2022 to 2024**. In particular, the median volatility difference rises sharply in **2022** and remains clearly high in **2023** and **2024**. This suggests that Autobahn stations not only exhibit higher fuel prices on average, but also show more short-term price fluctuations that are not fully explained by the controls in the panel regression.

Because the test is based on the **residuals from the panel regression**, we can not not simply conclude that raw Autobahn prices fluctuate more, but rather that **the unexplained part of fuel price volatility** is larger at Autobahn stations even after controlling for match-set (regional) structure, date effects, and brand-category differences.

A plausible substantive interpretation (similar to the interpretation of the AAutobahn premium) is that Autobahn stations may face a pricing environment with **weaker competitive pressure**, more rigid institutional structures (like the concession auctions), and potentially stronger exposure to market-wide shocks. The particularly large differences in **2022–2024** are consistent with the idea that periods of oil market disruption, such as the aftermath of the **war in Ukraine**, may have amplified this volatility gap. At the same time, these mechanisms are only possible explanations and are **not directly identified by the test itself**.

The later years should also be interpreted with some care, especially in 2026, since we only have the data for about 2 full months for far in this year.